<a href="https://colab.research.google.com/github/doomguy0991/dls_c/blob/main/C2%20-%20Improving%20Deep%20Neural%20Networks%20Hyperparameter%20tuning%2C%20Regularization%20and%20Optimization/Notes/Readme.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os

# Check if running in Google Colab
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("Setting up Colab environment...")

    # 1. Clone the repository
    repo_url = "https://github.com/doomguy0991/dls_c.git"
    # Only clone if the directory doesn't exist yet
    if not os.path.exists("/content/dls_c"):
        !git clone $repo_url /content/dls_c

    # 2. Change working directory to the notebook's location so relative paths for libraries work imports
    %cd "/content/dls_c/C2 - Improving Deep Neural Networks Hyperparameter tuning, Regularization and Optimization/Notes"

    print("Setup complete. You can now run the rest of the notebook.")
else:
    print("Running locally or out of Colab. No setup needed.")


### Improving Deep Neural Networks: Hyperparameter tuning, Regularization and Optimization

This is the second course of the deep learning specialization at [Coursera](https://www.coursera.org/specializations/deep-learning) which is moderated by [DeepLearning.ai](http://deeplearning.ai/). The course is taught by Andrew Ng.

#### Table of contents

* [Improving Deep Neural Networks: Hyperparameter tuning, Regularization and Optimization](#improving-deep-neural-networks-hyperparameter-tuning-regularization-and-optimization)
   * [Table of contents](#table-of-contents)
   * [Course summary](#course-summary)
   * [Practical aspects of Deep Learning](#practical-aspects-of-deep-learning)
      * [Train / Dev / Test sets](#train--dev--test-sets)
      * [Bias / Variance](#bias--variance)
      * [Basic Recipe for Machine Learning](#basic-recipe-for-machine-learning)
      * [Regularization](#regularization)
      * [Why regularization reduces overfitting?](#why-regularization-reduces-overfitting)
      * [Dropout Regularization](#dropout-regularization)
      * [Understanding Dropout](#understanding-dropout)
      * [Other regularization methods](#other-regularization-methods)
      * [Normalizing inputs](#normalizing-inputs)
      * [Vanishing / Exploding gradients](#vanishing--exploding-gradients)
      * [Weight Initialization for Deep Networks](#weight-initialization-for-deep-networks)
      * [Numerical approximation of gradients](#numerical-approximation-of-gradients)
      * [Gradient checking implementation notes](#gradient-checking-implementation-notes)
      * [Initialization summary](#initialization-summary)
      * [Regularization summary](#regularization-summary)
   * [Optimization algorithms](#optimization-algorithms)
      * [Mini-batch gradient descent](#mini-batch-gradient-descent)
      * [Understanding mini-batch gradient descent](#understanding-mini-batch-gradient-descent)
      * [Exponentially weighted averages](#exponentially-weighted-averages)
      * [Understanding exponentially weighted averages](#understanding-exponentially-weighted-averages)
      * [Bias correction in exponentially weighted averages](#bias-correction-in-exponentially-weighted-averages)
      * [Gradient descent with momentum](#gradient-descent-with-momentum)
      * [RMSprop](#rmsprop)
      * [Adam optimization algorithm](#adam-optimization-algorithm)
      * [Learning rate decay](#learning-rate-decay)
      * [The problem of local optima](#the-problem-of-local-optima)
   * [Hyperparameter tuning, Batch Normalization and Programming Frameworks](#hyperparameter-tuning-batch-normalization-and-programming-frameworks)
      * [Tuning process](#tuning-process)
      * [Using an appropriate scale to pick hyperparameters](#using-an-appropriate-scale-to-pick-hyperparameters)
      * [Hyperparameters tuning in practice: Pandas vs. Caviar](#hyperparameters-tuning-in-practice-pandas-vs-caviar)
      * [Normalizing activations in a network](#normalizing-activations-in-a-network)
      * [Fitting Batch Normalization into a neural network](#fitting-batch-normalization-into-a-neural-network)
      * [Why does Batch normalization work?](#why-does-batch-normalization-work)
      * [Batch normalization at test time](#batch-normalization-at-test-time)
      * [Softmax Regression](#softmax-regression)
      * [Training a Softmax classifier](#training-a-softmax-classifier)
      * [Deep learning frameworks](#deep-learning-frameworks)
      * [TensorFlow](#tensorflow)
   * [Extra Notes](#extra-notes)

## Course summary

Here are the course summary as its given on the course [link](https://www.coursera.org/learn/deep-neural-network):

> This course will teach you the "magic" of getting deep learning to work well. Rather than the deep learning process being a black box, you will understand what drives performance, and be able to more systematically get good results. You will also learn TensorFlow.
>
> After 3 weeks, you will:
> - Understand industry best-practices for building deep learning applications.
> - Be able to effectively use the common neural network "tricks", including initialization, L2 and dropout regularization, Batch normalization, gradient checking,
> - Be able to implement and apply a variety of optimization algorithms, such as mini-batch gradient descent, Momentum, RMSprop and Adam, and check for their convergence.
> - Understand new best-practices for the deep learning era of how to set up train/dev/test sets and analyze bias/variance
> - Be able to implement a neural network in TensorFlow.
>
> This is the second course of the Deep Learning Specialization.

## Practical aspects of Deep Learning

### Train / Dev / Test sets

- Its impossible to get all your hyperparameters right on a new application from the first time.
- So the idea is you go through the loop: `Idea ==> Code ==> Experiment`.
- You have to go through the loop many times to figure out your hyperparameters.
- Your data will be split into **three parts**:
  - **Training set**.       (Has to be the largest set)
  - Hold-out cross validation set / Development or **"dev" set**.
  - **Testing set**.
- You will try to **build a model upon training set** then **try to optimize hyperparameters on dev set** as much as possible. Then after your model is ready you try and **evaluate using the testing set**.
- so the trend on the ratio of splitting the models:
  - If size of the  dataset is 100 to 1000000  ==> 60/20/20
  - If size of the  dataset is 1000000  to INF  ==> 98/1/1 or  99.5/0.25/0.25
- The trend now gives the training data the biggest sets.
- Make sure the **dev** and **test set** are **coming from the same distribution**.
  - For example if cat training/dev pictures are from the web but the test pictures are from users cell phone they will mismatch. It is better to make sure that dev and test set are from the same distribution.
- The dev set rule is to try them on some of the good models you've created.
- Its OK to only have a dev set without a testing set. But a lot of people in this case call the dev set as the test set. A better terminology is to call it a dev set as its used in the development.


### Bias / Variance

- Bias / Variance techniques are Easy to learn, but difficult to master.
- So here the explanation of Bias / Variance:
  - If your model is underfitting (logistic regression of non linear data) it has a "high bias"
  - If your model is overfitting then it has a "high variance"
  - Your model will be alright if you balance the Bias / Variance
  - For more:
    - ![](https://github.com/doomguy0991/dls_c/blob/main/C2%20-%20Improving%20Deep%20Neural%20Networks%20Hyperparameter%20tuning%2C%20Regularization%20and%20Optimization/Notes/Images/01-_Bias_-_Variance.png?raw=1)
- Another idea to get the bias /  variance if you don't have a 2D plotting mechanism:
  - High variance (overfitting) for example:
    - Training error: 1%
    - Dev error: 11%
  - high Bias (underfitting) for example:
    - Training error: 15%
    - Dev error: 14%
  - high Bias (underfitting) && High variance (overfitting) for example:
    - Training error: 15%
    - Test error: 30%
  - Best:
    - Training error: 0.5%
    - Test error: 1%
  - These Assumptions came from that **human has 0% error** ->**(Optimal/Bayes error = 0%)**. If the problem isn't like that you'll need to use human error as baseline.

### Basic Recipe for Machine Learning

- If your algorithm has a **high bias**:
  - Try to make your NN bigger (size of hidden units, number of layers)
  - Try a different model that is suitable for your data.
  - Try to run it longer.
  - Different (advanced) optimization algorithms.
- If your algorithm has a **high variance**:
  - More data.
  - Try regularization.
  - Try a different model that is suitable for your data.
- You should **try the previous two points until** you have a **low bias and low variance**.
- In the older days before deep learning, there was a "Bias/variance tradeoff". But because now you have more options/tools for solving the bias and variance problem its really helpful to use deep learning.
- Training a bigger neural network never hurts.

### Regularization

- Adding regularization to NN will help it reduce variance (overfitting)
- L1 matrix norm:
  - `||W|| = Sum(|w[i,j]|)  # sum of absolute values of all w`
- L2 matrix norm because of arcane technical math reasons is called Frobenius norm:
  - `||W||^2 = Sum(|w[i,j]|^2)	# sum of all w squared`
  - Also can be calculated as `||W||^2 = W.T * W if W is a vector`
- Regularization for logistic regression:
  - The normal cost function that we want to minimize is: `J(w,b) = (1/m) * Sum(L(y(i),y'(i)))`
  - The L2 regularization version: `J(w,b) = (1/m) * Sum(L(y(i),y'(i))) + (lambda/2m) * Sum(|w[i]|^2)`
  - The L1 regularization version: `J(w,b) = (1/m) * Sum(L(y(i),y'(i))) + (lambda/2m) * Sum(|w[i]|)`
  - The L1 regularization version makes a lot of w values become zeros, which makes the model size smaller.
  - L2 regularization is being used much more often.
  - `lambda` here is the regularization parameter (hyperparameter)
- Regularization for NN:
  - The normal cost function that we want to minimize is:   
    `J(W1,b1...,WL,bL) = (1/m) * Sum(L(y(i),y'(i)))`

  - The L2 regularization version:   
    `J(w,b) = (1/m) * Sum(L(y(i),y'(i))) + (lambda/2m) * Sum((||W[l]||^2)`

  - We stack the matrix as one vector `(mn,1)` and then we apply `sqrt(w1^2 + w2^2.....)`

  - To do back propagation (old way):   
    `dw[l] = (from back propagation)`

  - The new way:   
    `dw[l] = (from back propagation) + lambda/m * w[l]`

  - So plugging it in weight update step:

    - ```
      w[l] = w[l] - learning_rate * dw[l]
           = w[l] - learning_rate * ((from back propagation) + lambda/m * w[l])
           = w[l] - (learning_rate*lambda/m) * w[l] - learning_rate * (from back propagation)
           = (1 - (learning_rate*lambda)/m) * w[l] - learning_rate * (from back propagation)
      ```

  - In practice this penalizes large weights and effectively limits the freedom in your model.

  - The new term `(1 - (learning_rate*lambda)/m) * w[l]`  causes the **weight to decay** in proportion to its size.


### Why regularization reduces overfitting?

Here are some intuitions:
  - Intuition 1:
     - If **`lambda` is too large** - a lot of **w's will be close to zeros** which will **make the NN simpler** (you can think of it as it would behave closer to logistic regression).
     - If **`lambda` is good enough** it will just **reduce some weights** that makes the neural network overfit.
  - Intuition 2 (with _tanh_ activation function):
     - If **`lambda` is too large**, **w's will be small** (close to zero) -> **z will become samll** {Z = WX+B} - will use the **linear part of the _tanh_ activation function**, so we will go from non linear activation to _roughly_ linear which would **make the NN a _roughly_ linear classifier**. ans as we know if the activation is liner then nomatter how deep the NN is it's ultimately a Linear regression
     - If `lambda` good enough it will just make some of _tanh_ activations _roughly_ linear which will prevent overfitting.
     
_**Implementation tip**_: if you implement gradient descent, one of the steps to **debug gradient descent** is to **plot the cost function J** as a function of the **number of iterations of gradient descent** and you want to see that the cost function J decreases **monotonically** after every elevation of gradient descent with regularization. If you plot the old definition of J (no regularization) then you might not see it decrease monotonically.

### Dropout Regularization

- In most cases Andrew Ng tells that he uses the L2 regularization.
- The dropout regularization **eliminates some neurons/weights on each iteration based on a probability.**
- Dropout prevents overfitting by ensuring the network does not become overly reliant on any singlevneuron or specific set of weights. It forces the network to learn redundant representations of   data.
- A most common technique to implement dropout is called **"Inverted dropout"**.
- <details>
  <summary>Click to expand!</summary>
  
  ### 1. What is Dropout?
  Dropout is a powerful regularization technique used to reduce variance and prevent overfitting in deep neural networks.
  *   During training, it randomly eliminates (sets to zero) a subset of neurons in a given layer on each iteration.
  *   The probability of a neuron remaining active is defined by the hyperparameter **`keep_prob`** (where $0 \le keep\_prob \le 1$). For example, if `keep_prob = 0.8`, 80% of the neurons stay active, and 20% are dropped.
  
  
  ### 2. Why Dropout Reduces Overfitting
  Even though we randomly shut down parts of the network, the model ultimately performs better on unseen data. This happens for three mathematical reasons:
  
  1.  **Prevents Feature Co-adaptation:** In a standard network, neurons can become highly dependent on the output of one or two specific neurons from the previous layer. With dropout, the network can never rely on any single input feature because that feature might be randomly dropped in the next iteration.
  2.  **Forces Weight Distribution:** Because a neuron cannot depend on a single input, it is forced to spread its weights across all of its inputs. This spreading out of weights naturally shrinks the magnitude of individual weights squared ($||W||^2$), producing an effect mathematically similar to L2 Regularization (Weight Decay).
  3.  **Ensemble Effect:** By dropping different combinations of neurons in every iteration, you are essentially training a large number of smaller, different sub-networks. The final model acts as a combined average (ensemble) of all these smaller networks, leading to a more generalized and robust output.
  
  ---
  
  ### 3. Implementation: "Inverted Dropout"
  The most common and computationally efficient way to implement dropout is called **Inverted Dropout**.
  
  **Implementation Code (for layer 3):**
  ```python
  keep_prob = 0.8   # 0 <= keep_prob <= 1
  l = 3             # targeting layer 3
  
  # 1. Create the boolean mask
  d3 = np.random.rand(a[l].shape[0], a[l].shape[1]) < keep_prob
  
  # 2. Apply the mask to eliminate neurons
  a3 = np.multiply(a3, d3)   
  
  # 3. Scale up the remaining activations (Inverted Dropout Step)
  a3 = a3 / keep_prob        
  ```
  
  #### Step-by-Step Breakdown:
  *   **The Mask ($d3$):** This is a matrix of the exact same shape as the activation matrix $A^{[3]}$. It contains boolean values (1 for True, 0 for False).
  *   **Element-wise Multiplication:** `np.multiply(a3, d3)` applies the mask. If a position in $d3$ is 0, the corresponding neuron in $a3$ becomes 0 (it is "dropped").
  
  ---
  
  ### 4. Deep Dive: Why do we divide by `keep_prob`? (The Scaling Step)
  The critical line of code `a3 = a3 / keep_prob` is what makes this technique "Inverted." This is done to solve a magnitude discrepancy between the Training Phase and the Test Phase.
  
  #### The Problem: Magnitude Discrepancy
  The linear input to the next layer is calculated as $Z^{[l+1]} = W^{[l+1]}A^{[l]} + b^{[l+1]}$.
  1.  **During Training:** If `keep_prob = 0.8`, 20% of the activations in $A^{[l]}$ are zeroed out. The total sum (signal strength) flowing into $Z^{[l+1]}$ decreases by roughly 20%.
  2.  **During Testing:** Dropout is turned **OFF** to maximize prediction accuracy. 100% of the neurons are active. Suddenly, $Z^{[l+1]}$ receives a signal strength that is 20% larger than what it was trained to handle.
  
  This shift in magnitude causes the activation functions in subsequent layers to operate in incorrect ranges, ruining the network's predictions.
  
  #### The Mathematical Solution: Expected Value
  In statistics, the **Expected Value** is the average value a variable takes over time.
  Let $a_i$ be an activation of a single neuron. With dropout, its expected value drops:
  $$E[a_{dropout}] = (keep\_prob \times a_i) + ((1 - keep\_prob) \times 0) = keep\_prob \times a_i$$
  
  To fix this, we artificially boost the 80% of remaining active neurons during training by dividing them by `keep_prob`.
  $$E[a_{scaled}] = keep\_prob \times \left( \frac{a_i}{keep\_prob} \right) = a_i$$
  
  **Conclusion:** By dividing by `keep_prob` during training, the expected value of the activations remains $a_i$. This ensures that when we switch to test time (using 100% of the neurons), the next layer sees the exact same magnitude of data it was trained on. We don't have to write extra scaling code for the testing phase.
  
  ---
  
  ### 5. Exam Checklist: Key Operational Rules
  *   **Iteration-Specific:** A new $d^{[l]}$ mask is generated for **every single pass** (iteration) through the data.
  *   **Consistent Across Propagation:** The exact same mask $d^{[l]}$ used during forward propagation must be cached and used during backpropagation to ensure gradients are only calculated for active neurons.
  *   **Test Time:** Do **not** use dropout at test/inference time. You want deterministic output without added noise. Because of the Inverted Dropout scaling step during training, no weight adjustments are necessary at test time.
</details>
- Code for Inverted dropout:

  ```python
  keep_prob = 0.8   # 0 <= keep_prob <= 1
  l = 3  # this code is only for layer 3
  # the generated number that are less than 0.8 will be dropped. 80% stay, 20% dropped
  d3 = np.random.rand(a[l].shape[0], a[l].shape[1]) < keep_prob

  # This element-wise multiplication ensures that if d3 is 0, the corresponding activation in a3 becomes 0.
  a3 = np.multiply(a3,d3)   

  # increase a3 to not reduce the expected value of output
  # (ensures that the expected value of a3 remains the same) - to solve the scaling problem
  a3 = a3 / keep_prob       
  ```
- Vector d[l] is used for forward and back propagation and is the same for them, but it is different for each iteration (pass) or training example.
- At test time we don't use dropout. If you implement dropout at test time - it would add noise to predictions.

### Understanding Dropout

- In the previous video, the intuition was that dropout randomly knocks out units in your network. So it's as if on every iteration you're working with a smaller NN, and so using a smaller NN seems like it should have a regularizing effect.
- Another intuition: can't rely on any one feature, so have to spread out weights.
- It's possible to show that **dropout has a similar effect to L2 regularization**.
- Dropout can have **different `keep_prob` per layer.**
- The input layer dropout has to be near 1 (or 1 - no dropout) because you don't want to eliminate a lot of features.
- If you're more worried about some layers overfitting than others, you can set a lower `keep_prob` for some layers than others. The downside is, this gives you even more hyperparameters to search for using cross-validation. One other alternative might be to have some layers where you apply dropout and some layers where you don't apply dropout and then just have one hyperparameter, which is a `keep_prob` for the layers for which you do apply dropouts.
- A lot of researchers are using **dropout with Computer Vision (CV)** because **they have a very big input size and almost never have enough data, so overfitting is the usual problem**. And dropout is a regularization technique to prevent overfitting.
- A downside of dropout is that the **cost function J is not well defined** and it will be **hard to debug** (plot J by iteration).
  - To solve that you'll need
    - to turn off dropout, set all the `keep_prob`s to 1,
    - and then run the code and check that it monotonically decreases J and
    - then turn on the dropouts again.

### Other regularization methods

#### Data augmentation:
  - For example in a computer vision data:
    - You can **flip all your pictures horizontally** this will give you **m more data instances.**
    - You could also **apply a random position and rotation to an image** to g**et more data.**
  - For example in OCR, you can impose random rotations and distortions to digits/letters.
  - New data obtained using this technique **isn't as good as the real independent data**, **but still can be used as a regularization technique**.

#### Early stopping:
  - In this technique we **plot the training set** and the **dev set cost together** for **each iteration**. At some iteration the dev set cost will stop decreasing and will start increasing.
  - We will pick the point at which the training set error and dev set error are best (lowest training cost with lowest dev cost).
  - We will take these parameters as the best parameters.
    - ![](https://github.com/doomguy0991/dls_c/blob/main/C2%20-%20Improving%20Deep%20Neural%20Networks%20Hyperparameter%20tuning%2C%20Regularization%20and%20Optimization/Notes/Images/02-_Early_stopping.png?raw=1)
  - Andrew prefers to use L2 regularization instead of early stopping because this technique simultaneously **tries to minimize the cost function** and **not to overfit** which **contradicts the orthogonalization approach** (will be discussed further).
  - But its advantage is that **you don't need to search a hyperparameter** like in other regularization approaches (like `lambda` in L2 regularization).

#### Model Ensembles:
  - Algorithm:
    - Train multiple independent models.
    - At test time average their results.
  - It can get you extra 2% performance.
  - It reduces the generalization error.
  - You can use some snapshots of your NN at the training ensembles them and take the results.


### Normalizing inputs

- If you normalize your inputs this will speed up the training process a lot.
- Normalization are going on these steps:
  1. Get the mean of the training set: `mean = (1/m) * sum(x(i))`
  2. Subtract the mean from each input: `X = X - mean`
     - This makes your inputs centered around 0.
  3. Get the variance of the training set: `variance = (1/m) * sum(x(i)^2)`
  4. Normalize the variance. `X /= variance`
- These steps should be applied to training, dev, and testing sets (but using mean and variance of the train set).
- Why normalize?
  - If we don't normalize the inputs our cost function will be deep and its shape will be inconsistent (elongated) then optimizing it will take a long time.
  - But if we normalize it the opposite will occur. The shape of the cost function will be consistent (look more symmetric like circle in 2D example) and we can use a larger learning rate alpha - the optimization will be faster.


### Vanishing / Exploding gradients

- The Vanishing / Exploding gradients occurs when your ***derivatives become very small or very big.***
- To understand the problem, suppose that we have a deep neural network with number of layers L, and all the activation functions are **linear** and each `b = 0`
  - Then:   
    ```
    Y' = W[L]W[L-1].....W[2]W[1]X
    ```
  - Then, if we have 2 hidden units per layer and x1 = x2 = 1, we result in:

    ```
    if W[l] = [1.5   0]
              [0   1.5] (l != L because of different dimensions in the output layer)
    Y' = W[L] [1.5  0]^(L-1) X = 1.5^L 	# which will be very large
              [0  1.5]
    ```
    ```
    if W[l] = [0.5  0]
              [0  0.5]
    Y' = W[L] [0.5  0]^(L-1) X = 0.5^L 	# which will be very small
              [0  0.5]
    ```
- The last example explains that the **activations (and similarly derivatives) will be decreased/increased exponentially as a function of number of layers.**
- So If W > I (Identity matrix) the activation and gradients will explode.
- And If W < I (Identity matrix) the activation and gradients will vanish.
- Recently Microsoft trained 152 layers (ResNet)! which is a really big number. With such a deep neural network, if your activations or gradients increase or decrease exponentially as a function of L, then these values could get really big or really small. And this makes training difficult, especially if your gradients are exponentially smaller than L, then gradient descent will take tiny little steps. It will take a long time for gradient descent to learn anything.
- There is a partial solution that doesn't completely solve this problem but it helps a lot - careful choice of how you initialize the weights (next video).


### Weight Initialization for Deep Networks

- A *partial solution to the **Vanishing / Exploding gradients*** in NN is better or more careful choice of the random initialization of weights
- In a single neuron (Perceptron model): `Z = w1x1 + w2x2 + ... + wnxn`
  - So if `n_x` is large we want `W`'s to be smaller to not explode the cost.
- So it turns out that we need the variance which equals `1/n_x` to be the range of `W`'s
- So lets say when we initialize `W`'s like this (better to use with `tanh` activation):   
  ```
  np.random.rand(shape) * np.sqrt(1/n[l-1])
  ```
  or variation of this (Bengio et al.):   
  ```
  np.random.rand(shape) * np.sqrt(2/(n[l-1] + n[l]))
  ```
- Setting initialization part inside sqrt to `2/n[l-1]` for `ReLU` is better:   
  ```
  np.random.rand(shape) * np.sqrt(2/n[l-1])
  ```
- Number 1 or 2 in the neumerator can also be a hyperparameter to tune (but not the first to start with)
- This is one of the best way of partially solution to Vanishing / Exploding gradients (ReLU + Weight Initialization with variance) which will help gradients not to vanish/explode too quickly
- The initialization in this video is called "He Initialization / Xavier Initialization" and has been published in 2015 paper.

<details>
  <summary>Click to expand!</summary>
  
To understand the relationship between variance, scaling, and the size of the weight $W$, we have to look at the **balance** required to keep the signal $Z$ stable.

The core relationship is **inverse**: as the number of inputs ($n$) gets **bigger**, the individual weights ($W$) must get **smaller**.

---

### 1. The Relationship Logic
The equation for a neuron is $Z = \sum_{i=1}^{n} w_i x_i$.

If we assume the inputs $x$ have a variance of 1, the variance of the sum $Z$ is roughly:
$$\text{Var}(Z) = n \times \text{Var}(W)$$

*   **If $n$ (number of inputs) is LARGE:** You are adding thousands of numbers together. To prevent the total sum $Z$ from blowing up to infinity (Exploding Gradient), each individual $w_i$ must be very tiny.
*   **If $n$ (number of inputs) is SMALL:** You only have a few numbers. If the weights are too small, the signal $Z$ will be so weak that it disappears (Vanishing Gradient). Therefore, the weights can afford to be larger.

### 2. How the Math forces $W$ to be smaller
We want the variance of $Z$ to be **1** so that the signal remains "standardized" as it passes through the layers.

To get $\text{Var}(Z) = 1$, we set:
$$1 = n \times \text{Var}(W) \implies \text{Var}(W) = \frac{1}{n}$$

Because we initialize weights using a standard normal distribution (which has a variance of 1), we multiply it by a **scaling factor** to change its variance to $1/n$.
Since the scaling factor is applied to the values (Standard Deviation), and $\text{Standard Deviation} = \sqrt{\text{Variance}}$, we get:
$$\text{Weight Scaling Factor} = \sqrt{\frac{1}{n}}$$

### 3. Summary Table: The Balancing Act

| If the number of inputs ($n$) is... | The Variance of $W$ ($\frac{1}{n}$) becomes... | The actual weights ($W$) become... | Impact on $Z = WX+B$ |
| :--- | :--- | :--- | :--- |
| **Bigger** (e.g., 1000) | **Smaller** (0.001) | **Smaller** (closer to 0) | $Z$ stays in a healthy range instead of exploding. |
| **Smaller** (e.g., 10) | **Bigger** (0.1) | **Bigger** (further from 0) | $Z$ stays strong enough to be detected by the next layer. |

### 4. Why does this matter for "Saturation"?
This is directly linked to your earlier question about **Tanh** and **Sigmoid**.
*   If you have 1,000 inputs and you *don't* scale the weights down, $Z$ will become a very large number (like $+50$ or $-50$).
*   On the Tanh/Sigmoid curve, $+50$ is in the "flat" zone where the slope (derivative) is **zero**.
*   If the derivative is zero, the network stops learning.

**By scaling $W$ to be smaller when $n$ is bigger, you are mathematically forcing $Z$ to stay near the "steep" part of the activation function where the slope is high and learning is fast.**

---

In short: The variance $1/n$ is the "budget" for the total signal. If you have more guests (inputs), everyone gets a smaller slice of the pie (weight magnitude) to keep the total pie size (variance of $Z$) the same.

Does the math behind why we use the square root ($\sqrt{1/n}$) for the code vs. $1/n$ for the variance make sense now?
</details>

### Gradient Checking (Numerical Differentiation)

### 1. What is Gradient Checking?
Gradient checking is a "sanity check" used to verify that your manually coded **Backpropagation** (the analytical gradient) is mathematically correct. It uses a numerical approximation of the derivative to ensure your code isn't suffering from subtle mathematical bugs.

### 2. Why & When is it needed?
*   **The Problem:** Backpropagation is complex. A tiny typo (like a misplaced minus sign or a missing term) can lead to a model that "runs" without errors but never actually converges to the optimal solution.
*   **The Solution:** You compare your code’s output ($d\theta_{approx}$) with a numerical "ground truth" ($d\theta_{grad}$).
*   **When to use:**
    *   Only during **debugging/development**.
    *   Never during actual training (it is extremely slow because it requires computing the cost function twice for every single parameter).

---

### 3. What are we basically doing?
We are using the geometric definition of a derivative. If we want to find the slope of the cost function $J$ at a specific point $\theta$, we move a tiny amount $\epsilon$ (epsilon) away from $\theta$ and see how much the cost changes.

---

### 4. Method Comparison: One-Sided vs. Two-Sided
To find the gradient numerically, we have two options:

#### Option A: One-Sided Difference
This looks at the change from the current point to a point slightly ahead.
$$\text{Formula: } \frac{J(\theta + \epsilon) - J(\theta)}{\epsilon}$$
*   **Accuracy:** Moderate. It follows the standard limit definition of a derivative.

#### Option B: Two-Sided Difference (The "Andrew Ng" Way)
This "brackets" the point $\theta$ by looking slightly behind and slightly ahead.
$$\text{Formula: } \frac{J(\theta + \epsilon) - J(\theta - \epsilon)}{2\epsilon}$$
*   **Accuracy:** **Significantly Higher.** This is the preferred method for Gradient Checking.

---

### 5. Why is Two-Sided "Better"? (The Order of Error)
In numerical analysis, "better" means the **error is smaller**. We use Taylor Series Expansion to prove why the Two-Sided version is superior.

#### The Error of One-Sided: $O(\epsilon)$
When we expand the math for the one-sided formula, the leftover error term is proportional to $\epsilon$.
*   If $\epsilon = 10^{-7}$, your error is roughly **$0.0000001$**.

#### The Error of Two-Sided: $O(\epsilon^2)$
In the two-sided formula, the Taylor expansion shows that the first-order error terms actually **cancel each other out**, leaving an error term proportional to $\epsilon^2$.
*   If $\epsilon = 10^{-7}$, your error is $(10^{-7})^2 = 10^{-14}$.
*   Your error becomes **$0.00000000000001$**.

> **Summary:** The Two-Sided formula is $1,000,000$ times more accurate than the One-Sided formula when using a small epsilon. This precision is necessary because when checking gradients, we need to distinguish between a "tiny numerical rounding error" and a "math bug in our code."

---
### 6. Implementation
1.  **Reshape Parameters:** Take all your weight matrices $W^{[L]}$ and bias vectors $b^{[L]}$ and concatenate (reshape) them into one large vector called $\theta$.
2.  **Reshape Gradients:** Similarly, take all your computed gradients $dW^{[L]}$ and $db^{[L]}$ and concatenate them into one large vector called $d\theta$.
3.  **Define the Cost Function:** You now have a function $J(\theta)$ that takes this single large vector and returns the cost.

#### The Approximation Algorithm
For each element $i$ in the vector $\theta$:
```python
epsilon = 1e-7   # A very small value
for i in range(len(theta)):
    # Create two temporary copies to shift a single parameter
    theta_plus = np.copy(theta)
    theta_plus[i] = theta_plus[i] + epsilon
    
    theta_minus = np.copy(theta)
    theta_minus[i] = theta_minus[i] - epsilon
    
    # Calculate the numerical gradient for this specific parameter
    d_theta_approx[i] = (J(theta_plus) - J(theta_minus)) / (2 * epsilon)
```
---
### 7. How to interpret the results?
After calculating the numerical gradient ($d\theta_{approx}$) and your backprop gradient ($d\theta$), you compute the **relative difference**:

$$\text{Difference} = \frac{\|d\theta_{approx} - d\theta\|_2}{\|d\theta_{approx}\|_2 + \|d\theta\|_2}$$

| Difference Value | Conclusion |
| :--- | :--- |
| **$10^{-7}$ or smaller** | **Great!** Your backprop is likely perfect. |
| **$10^{-5}$** | **Careful.** There might be a subtle bug; double-check the math. |
| **$10^{-3}$ or larger** | **Bug Detected.** Something is wrong in your backpropagation code. |

---

**Practical Tip for your Research:** Most modern frameworks like PyTorch or JAX have a built-in `gradcheck` function that uses this exact Two-Sided logic. When you start writing custom layers for your Vision-Language Model (VLM) thesis, running a `gradcheck` on your new layer is the first thing you should do!

Since you're looking into Computer Vision research, are you planning to implement any custom loss functions for your thesis, or are you sticking to standard ones for now?